In [ ]:
import os
import dotenv
import pandas as pd
import pyodbc
import datetime
from zoneinfo import ZoneInfo
import matplotlib.pyplot as plt
import seaborn as sns

dotenv.load_dotenv()


# Connection string using the specific InterSystems ODBC driver
# Note: Ensure the driver name exactly matches what is installed on your client
# Updated with the correct driver name from your system
connection_string = (
    f"DRIVER={{InterSystems IRIS ODBC35}};"
    f"SERVER={os.getenv('IRIS_SERVER')};"
    f"PORT={os.getenv('IRIS_PORT')};"
    f"DATABASE={os.getenv('IRIS_NAMESPACE')};"
    f"UID={os.getenv('IRIS_USERNAME')};"
    f"PWD={os.getenv('IRIS_PASSWORD')}"
)

In [ ]:
past_date_str = "2024-01-01 00:00:00"
past_datetime = datetime.datetime.strptime(past_date_str, "%Y-%m-%d %H:%M:%S")
current_datetime = datetime.datetime.now(tz=ZoneInfo(os.getenv("TZ")))
datetime_now_str = current_datetime.strftime("%Y-%m-%d %H:%M:%S")

with pyodbc.connect(connection_string) as cnxn:
    PointSamples_df = pd.read_sql(f"SELECT * FROM MLpipeline.PointSamples WHERE datetime >= '{past_datetime}' AND datetime <= '{datetime_now_str}'", cnxn)
    points_w_groundtruth = PointSamples_df[PointSamples_df.y.notna()].ID
    Predictions_df = pd.read_sql(f"SELECT * FROM MLpipeline.Predictions WHERE pointsidforeignkey IN {tuple(points_w_groundtruth)}", cnxn)

In [ ]:
points_w_groundtruth

In [ ]:
eachmodelspred = Predictions_df.pivot_table(index="PointsIDForeignKey", columns="Modelrunid", values="YPred", aggfunc="mean")
merged_df = pd.merge(PointSamples_df, eachmodelspred, left_on="ID", right_on="PointsIDForeignKey", how="left")
modelnames = [col for col in merged_df.columns if col not in PointSamples_df.columns]

# Grained Comparison Plot

In [ ]:
plt.figure(figsize=(12, 6))
for colname in modelnames + ["y"]:
    sns.scatterplot(data=merged_df, x="x", y=colname, label=colname)
plt.title("Predictions Over Time by Model Run")
plt.xlabel("Datetime")
plt.ylabel("Y")
plt.legend(title="Model Run ID")
plt.xticks()
plt.grid()
plt.tight_layout()

In [ ]:
models_last_datetime = Predictions_df.groupby("Modelrunid").agg({"datetime":max})
latest_model = models_last_datetime["datetime"].idxmax()
latest_active_predictions = Predictions_df[Predictions_df["Modelrunid"] == latest_model].groupby("PointsIDForeignKey").agg({"YPred": "mean", "InferenceTime": "mean"}).reset_index()
latest_active_predictions = pd.merge(latest_active_predictions, PointSamples_df[["ID", "y"]], left_on="PointsIDForeignKey", right_on="ID", how="left")


In [ ]:
latest_active_predictions

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(latest_active_predictions["y"], latest_active_predictions["YPred"])
r2 = r2_score(latest_active_predictions["y"], latest_active_predictions["YPred"])
print(f"Mean Absolute Error: {mae}")
print(f"R-squared: {r2}")
